# **ETL Excel to PostgreSQL using Docker**

In [ ]:
# Import required library

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date # to define production date, create label data that we want to extract
from sqlalchemy import create_engine

In [ ]:
# Load data from Excel file

df = pd.read_excel(r"E:\Self Learning\03. Learn Python\Python_Workspace\ETL_Excel_with_Pandas\Shift Report Tipe A Februari 2026.xlsx", 
                   sheet_name="01", 
                   skiprows=9,
                   usecols="A:O",
                   nrows=25)

In [ ]:
# Drop unnecessary columns

df.drop(df.columns[[1,3,4,5]],axis=1, inplace=True) 
# axis to explain we are working on columns axis=1
# inplace=True to save the changes in df, if False then changes are not saved in df

df.columns

Index(['Unnamed: 0', 'Unnamed: 2', 'Product', 'D.Time', '122T...', 'Product.1',
       'D.Time.1', '122T....1', 'Product.2', 'D.Time.2', '122T....2'],
      dtype='object')

In [ ]:
# Rename columns to make easier to work with

df.rename(columns={"Unnamed: 0" : "Section",
                   "Unnamed: 2" : "Product",
                   "Product" : "Shift_2",
                   "D.Time" : "Downtime_shift_2",
                   "122T..." : "Tangki_shift_2",
                   "Product.1" : "Shift_3",
                   "D.Time.1" : "Downtime_shift_3",
                   "122T....1" : "Tangki_shift_3",
                   "Product.2" : "Shift_1",
                   "D.Time.2" : "Downtime_shift_1",
                   "122T....2" : "Tangki_shift_1"}, inplace=True)

In [ ]:
# Change column names to lowercase for consistency

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_") # optional to replace spaces with underscores for easier access
df.columns = df.columns.str.strip() # optional to remove leading and trailing spaces from column names (similar to trim in SQL)
df.head()

,section,product,shift_2,downtime_shift_2,tangki_shift_2,shift_3,downtime_shift_3,tangki_shift_3,shift_1,downtime_shift_1,tangki_shift_1
0,101,Feed Stream (C16-18 Raw Ester),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Oleic Intermediate Stream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Heavy Phase Fraction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,102,Feed Stream (Hydrogenated C16-18),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Feed Stream (Hydrogenated Process Stock),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Fill in empty values (NaN)

df['section'] = df['section'].ffill() # to fill empty values with the previous value (forward fill)
df.head()

,section,product,shift_2,downtime_shift_2,tangki_shift_2,shift_3,downtime_shift_3,tangki_shift_3,shift_1,downtime_shift_1,tangki_shift_1
0,101,Feed Stream (C16-18 Raw Ester),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,101,Oleic Intermediate Stream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,101,Heavy Phase Fraction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,102,Feed Stream (Hydrogenated C16-18),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,102,Feed Stream (Hydrogenated Process Stock),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Create new column to categorize activity type based on product name

df['activity'] = np.where(df['product'].str.contains("Feed", case=False, na=False), "Feed", "Production")
df['production_dates'] = pd.to_datetime("2026-02-01")
df.head()

,section,product,shift_2,downtime_shift_2,tangki_shift_2,shift_3,downtime_shift_3,tangki_shift_3,shift_1,downtime_shift_1,tangki_shift_1,activity,production_dates
0,101,Feed Stream (C16-18 Raw Ester),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Feed,2026-02-01
1,101,Oleic Intermediate Stream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Production,2026-02-01
2,101,Heavy Phase Fraction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Production,2026-02-01
3,102,Feed Stream (Hydrogenated C16-18),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Feed,2026-02-01
4,102,Feed Stream (Hydrogenated Process Stock),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Feed,2026-02-01


In [ ]:
# Performing melt on shift columns to transform data from wide format to long format for easier analysis and database insertion

df_melted_prod = df.melt(id_vars=['section', 'activity', 'production_dates'],
                    value_vars=['shift_1', 'shift_2', 'shift_3'],
                    var_name='shift_category',
                    value_name='shift_value')

df_melted_prod['shift_value'] = df_melted_prod['shift_value'].fillna(0)
df_melted_prod['shift_category'] = df_melted_prod['shift_category'].str.replace("shift_", "Shift ")

df_melted_dt = df.melt(id_vars=['section', 'activity', 'production_dates'],
                    value_vars=['downtime_shift_1', 'downtime_shift_2', 'downtime_shift_3'],
                    var_name='shift_category',
                    value_name='downtime_value')

df_melted_dt['downtime_value'] = df_melted_dt['downtime_value'].fillna(0)
df_melted_dt['shift_category'] = df_melted_dt['shift_category'].str.replace("downtime_shift_", "Shift ")

df_melted_tangki = df.melt(id_vars=['section', 'activity', 'production_dates'],
                    value_vars=['tangki_shift_1', 'tangki_shift_2', 'tangki_shift_3'],
                    var_name='shift_category',
                    value_name='tangki_value')

df_melted_tangki['tangki_value'] = df_melted_tangki['tangki_value'].fillna("-")
df_melted_tangki['shift_category'] = df_melted_tangki['shift_category'].str.replace("tangki_shift_", "Shift ")

df_melted_tangki


,section,activity,production_dates,shift_category,tangki_value
0,101,Feed,2026-02-01,Shift 1,-
1,101,Production,2026-02-01,Shift 1,-
2,101,Production,2026-02-01,Shift 1,-
3,102,Feed,2026-02-01,Shift 1,-
4,102,Feed,2026-02-01,Shift 1,-
...,...,...,...,...,...
64,104,Feed,2026-02-01,Shift 3,-
65,104,Feed,2026-02-01,Shift 3,-
66,104,Feed,2026-02-01,Shift 3,-
67,104,Production,2026-02-01,Shift 3,T129


In [ ]:
# to merge df_melted_prod and df_melted_dt based on the same columns (section, activity, production_dates, shift_category)
df_finalize = df_melted_prod.merge(df_melted_dt, on=['section', 'activity', 'production_dates', 'shift_category'])

# to merge df_finalize and df_melted_tangki based on the same columns (section, activity, production_dates, shift_category)
df_finalize = df_finalize.merge(df_melted_tangki, on=['section', 'activity', 'production_dates', 'shift_category'])

df_finalize

,section,activity,production_dates,shift_category,shift_value,downtime_value,tangki_value
0,101,Feed,2026-02-01,Shift 1,0.0,0.0,-
1,101,Production,2026-02-01,Shift 1,0.0,0.0,-
2,101,Production,2026-02-01,Shift 1,0.0,0.0,-
3,101,Production,2026-02-01,Shift 1,0.0,0.0,-
4,101,Production,2026-02-01,Shift 1,0.0,0.0,-
...,...,...,...,...,...,...,...
1180,104,Production,2026-02-01,Shift 3,0.0,0.0,-
1181,104,Production,2026-02-01,Shift 3,0.0,0.0,T129
1182,104,Production,2026-02-01,Shift 3,0.0,0.0,-
1183,104,Production,2026-02-01,Shift 3,0.0,0.0,T129


In [ ]:
# Create container PostgreSQL using Docker
# docker run -d --name postgres-db -e POSTGRES_USER=admin -e POSTGRES_PASSWORD=admin123 -e POSTGRES_DB=mydatabase -p 5433:5432 postgres


**docker run -d --name postgres-db -e POSTGRES_USER=admin -e POSTGRES_PASSWORD=admin123 -e POSTGRES_DB=mydatabase -p 5433:5432 postgres**

Penjelasan Docker
* docker run--> It tells Docker to create a new container from an image and start it up
* run --> If you don't have the image on your computer yet, Docker will automatically download (pull) it from Docker Hub
* -d --> (detached mode), it runs the container in the background
* --name --> menamakan nama container kita
* postgres-db --> give your container name
* -e POSTGRES_USER --> environment to sets the superuser name for PostgreSQL
* -e POSTGRES_PASSWORD --> environment to sets the password
* -e POSTGRES_DB --> environment to sets database name in PostgreSQL
* -p 5433:5432 --> port 5433 is the port on your physical machine (Host) and port 5432 is the port inside the container. Why using 5433? Because there is already have existing PostgreSQL instalation outside Container
* postgres --> is the name of the image to use

Then execute script above to be run on terminal

In [ ]:
########################################################################################
# WARNING: Make sure the values ​​below are not published publicly or uploaded to the repository,
# because this is sensitive information that could be used to access our database.
# For larger projects, it's best to use environment variables or configuration files that aren't uploaded to the repository to store this information.
########################################################################################

DB_USER = "admin"
DB_PASS = "admin123"
DB_HOST = "localhost"
DB_PORT = "5433" # default port for PostgreSQL is 5432, but since there's an existing PostgreSQL instance using port 5432, we map it to 5433 on the host
DB_NAME = "mydatabase"

In [ ]:
# Create engine to connect to PostgreSQL database using SQLAlchemy

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
# Saving data to a table in the PostgreSQL database using the to_sql method from pandas

df_finalize.to_sql(name="report_type_a", con=engine, if_exists="replace", index=False)

185